# 🎯 TexTyle Precision Search Server on Google Colab

`precision_search_main.py` 를 Colab GPU(T4 등)에서 실행하고, ngrok 으로 공개 URL을 만들어
로컬 노트북보다 빠르게 모바일 앱에서 호출할 수 있게 합니다.

**필요한 것:**
1. Colab GPU 런타임 (메뉴: 런타임 → 런타임 유형 변경 → T4 GPU)
2. Supabase URL/KEY, Gemini API KEY  
3. ngrok 계정 무료 토큰 (https://dashboard.ngrok.com/get-started/your-authtoken)
4. 로컬에서 `precision_search_main.py` 업로드 (왼쪽 파일 패널에 드래그)

**흐름 요약:**
- 셀1: GPU 확인
- 셀2: 의존성 설치 (~5분, BLIP-2 ViT-G ~14GB 다운로드는 첫 실행 시)
- 셀3: 환경변수 설정 (Colab Secrets 또는 직접 입력)
- 셀4: precision_search_main.py 패치 (Colab 환경용)
- 셀5: ngrok 터널 생성
- 셀6: 서버 실행 (백그라운드) + 헬스체크
- 셀7: 종료 (필요시)

## 셀 1 — GPU 확인

T4(15GB) 이상이면 4-bit nf4 로 BLIP-2 ViT-G 가 여유롭게 들어갑니다.

In [ ]:
!nvidia-smi

## 셀 2 — 의존성 설치

Colab 기본 환경에 추가로 필요한 패키지만 설치합니다.  
`bitsandbytes` 가 4-bit 양자화 핵심 — 설치 실패 시 자동으로 bf16 fallback 동작합니다.

In [ ]:
%pip install -q \
    fastapi "uvicorn[standard]" python-multipart \
    open_clip_torch supabase python-dotenv pillow httpx \
    google-genai \
    bitsandbytes accelerate \
    pyngrok nest-asyncio

## 셀 3 — 환경변수 설정

**권장:** Colab 좌측 자물쇠 아이콘(🔑 비밀)에 다음을 등록 후 Notebook access ON
- `SUPABASE_URL`
- `SUPABASE_KEY`
- `GEMINI_API_KEY`
- `NGROK_AUTH_TOKEN`

**대안:** 아래 셀에서 직접 입력 (단, 노트북 공유 시 노출 위험).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["SUPABASE_URL"]    = userdata.get("SUPABASE_URL")
    os.environ["SUPABASE_KEY"]    = userdata.get("SUPABASE_KEY")
    os.environ["GEMINI_API_KEY"]  = userdata.get("GEMINI_API_KEY")
    os.environ["NGROK_AUTH_TOKEN"] = userdata.get("NGROK_AUTH_TOKEN")
    print("✅ Colab Secrets에서 로드 완료")
except Exception as e:
    print(f"⚠️ Secrets 로드 실패: {e}")
    print("아래 변수에 직접 값을 채워서 다시 실행하세요.")
    # os.environ["SUPABASE_URL"]    = "https://xxx.supabase.co"
    # os.environ["SUPABASE_KEY"]    = "eyJxxx..."
    # os.environ["GEMINI_API_KEY"]  = "AIzaSyxxx"
    # os.environ["NGROK_AUTH_TOKEN"] = "2abc..."

# 선택 옵션
os.environ.setdefault("GEMINI_MODEL_NAME", "gemini-2.5-flash")
os.environ.setdefault("STAGE1_COUNT", "200")
os.environ.setdefault("STAGE2_TOP", "20")

# 확인 (KEY 는 일부만 표시)
for k in ["SUPABASE_URL", "SUPABASE_KEY", "GEMINI_API_KEY", "NGROK_AUTH_TOKEN"]:
    v = os.environ.get(k, "")
    print(f"  {k:18s} = {v[:12]+'…' if v else '(미설정)'}")

## 셀 4 — `precision_search_main.py` 업로드 + Colab 패치

**먼저** Colab 좌측 파일 패널에 `precision_search_main.py` 를 드래그 업로드하세요.  
그 다음 이 셀이 다음 2가지를 자동 수행합니다:
1. `.env` 대신 환경변수에서 직접 읽도록 보장 (이미 환경변수 셋팅됨)
2. uvicorn `__main__` 가드를 비활성화 (셀에서 import 후 직접 구동)

In [ ]:
import os, pathlib

src = pathlib.Path("precision_search_main.py")
assert src.exists(), "❌ precision_search_main.py 가 없습니다. 좌측 파일 패널에 업로드하세요."

text = src.read_text(encoding="utf-8")

# .env 파일이 없어도 환경변수만으로 동작하도록 — load_dotenv 자체는 그대로 두어도 OK
# (이미 환경변수가 셋되어 있으면 load_dotenv 가 덮어쓰지 않음)

# Colab 에서는 BASE_DIR/.env 가 없을 수 있으므로 빈 파일이라도 만들어 줌
pathlib.Path(".env").touch(exist_ok=True)

print(f"✅ precision_search_main.py 준비 완료 ({len(text):,} bytes)")

## 셀 5 — ngrok 터널 생성

ngrok 무료 토큰 1개로 1개 터널을 항상 같은 도메인 없이 매번 새로운 URL이 발급됩니다.  
발급된 `https://xxx.ngrok-free.app` 을 앱의 `SERVER_BASE_URL` 에 넣어주세요.

In [ ]:
from pyngrok import ngrok, conf
import os

token = os.environ.get("NGROK_AUTH_TOKEN")
assert token, "❌ NGROK_AUTH_TOKEN 미설정. 셀 3 다시 실행."

conf.get_default().auth_token = token

# 기존 터널이 있으면 정리
for t in ngrok.get_tunnels():
    try: ngrok.disconnect(t.public_url)
    except: pass

public_url = ngrok.connect(8002, "http").public_url
print("\n" + "=" * 60)
print(f"🌐 ngrok 공개 URL : {public_url}")
print(f"📱 앱에서 사용할 base: {public_url}")
print("=" * 60)
print("\n앱의 precision_search.tsx 에서:")
print(f'  const SERVER_BASE_URL = "{public_url}";')
print("으로 바꾸면 됩니다.")

## 셀 6 — 서버 실행 (백그라운드) + 헬스체크

처음 실행 시 모델 다운로드 때문에 2-5분 소요됩니다 (BLIP-2 ViT-G ~14GB, fashionSigLIP ~2GB).
두 번째 셀 실행부터는 캐시 사용으로 ~30초 안에 시작됩니다.

In [ ]:
import subprocess, time, requests, sys, os

# 기존 프로세스 종료
subprocess.run(["pkill", "-f", "precision_search_main"], stderr=subprocess.DEVNULL)
time.sleep(1)

# stdout/stderr 를 파일로 (실시간 tail 가능)
log_path = "/content/precision_server.log"
log_f = open(log_path, "w")

env = os.environ.copy()
proc = subprocess.Popen(
    [sys.executable, "precision_search_main.py"],
    stdout=log_f, stderr=subprocess.STDOUT, env=env,
)
print(f"🚀 서버 시작 (PID={proc.pid}). 로그: {log_path}")
print("   초기 모델 로딩 중... 최대 5분 정도 걸립니다.")

# 헬스체크 (최대 5분 = 300초 대기)
started = False
for i in range(60):
    time.sleep(5)
    try:
        r = requests.get("http://localhost:8002/health", timeout=3)
        if r.status_code == 200:
            print(f"\n✅ 서버 정상! ({(i+1)*5}초)")
            print(r.json())
            started = True
            break
    except Exception:
        pass
    if proc.poll() is not None:
        print(f"\n❌ 서버 프로세스 종료됨 (exit={proc.returncode}). 로그 확인:")
        break
    print(f"  ⏳ {(i+1)*5}s 대기 중...")

if not started:
    print("\n--- 최근 로그 ---")
    with open(log_path) as f:
        print(f.read()[-3000:])

## 셀 6.5 — (선택) 로그 라이브 보기

In [ ]:
!tail -n 50 /content/precision_server.log

## 셀 7 — (선택) 종료

Colab 세션을 그대로 두면 서버가 계속 동작합니다.  
정리하려면 아래 셀을 실행하세요.

In [ ]:
import subprocess
from pyngrok import ngrok

subprocess.run(["pkill", "-f", "precision_search_main"], stderr=subprocess.DEVNULL)
for t in ngrok.get_tunnels():
    try: ngrok.disconnect(t.public_url)
    except: pass
ngrok.kill()
print("🧹 정리 완료")